In this notebook, we compare the morphological parameters from SExtractor in the R21 catalogues with actual galaxy images made with MUSE cubes. The goal is to figure out whether these parameters are useful for estimating galax sizes, which in turn can be used to estimate the degree to which cluster members contaminate targets

In [ ]:
# Load megatable
from astropy.io import fits
from astropy.table import Table

megatab = Table(fits.open(f"./megatables/lae_megatab_1fwhm_opt.fits")[1].data)

In [ ]:
# Generate a muse white light image centered on sources of interest 
megatab.sort("SNRR", reverse=True) # sort by SNRR to get the brightest sources first
# Restrict to a single cluster to speed up processing time
cluster_of_interest = "MACS0416NE"
megatab_interest = megatab[megatab["CLUSTER"] == cluster_of_interest]

N_sources = 3

sources_of_interest = megatab_interest[:N_sources]

sources_of_interest # display the sources of interest

In [ ]:
# For each source, make a muse whte light image cutout centered on it, and plot an ellipse showing the SExtractor morphological
# parameters (A_WORLD, B_WORLD, THETA_J2000) from the R21 tables for nearby sources
from tangelo.quality_control import find_nearby_sources
from tangelo.image_processing import make_muse_img

import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from matplotlib.colors import PowerNorm

from astropy.coordinates import SkyCoord
from astropy import units as u
import numpy as np

cutout_size = 10 # arcseconds

for source in sources_of_interest:
    source_id = source["iden"]
    source_ra = source["RA"]
    source_dec = source["DEC"]

    # Make a muse white light image cutout centered on the source
    muse_img = make_muse_img(source, cutout_size * 2, 6000, 1000)

    fig, ax = plt.subplots(figsize=(5, 5))
    # Set reasonable stretch limits for the white light image
    vmin = 0
    vmax = np.percentile(muse_img.data, 99) # Use the 99th percentile to avoid outliers dominating the stretch
    ax.imshow(muse_img.data, origin="lower", cmap="gray", norm=PowerNorm(0.5, vmin=vmin, vmax=vmax))

    # Find nearby sources in the R21 tables
    nearby_sources = find_nearby_sources(source, maxdist=cutout_size)

    print(f"Found {len(nearby_sources)} nearby sources for source ID {source_id}")

    # Plot an ellipse for each nearby source using the SExtractor morphological parameters
    for nearby_source in nearby_sources:
        nearby_source_id = nearby_source["iden"]
        ra = nearby_source["RA"]
        dec = nearby_source["DEC"]
        a_world = nearby_source["A_WORLD"] * np.nan # In degrees
        b_world = nearby_source["B_WORLD"] * np.nan # in degrees
        theta_j2000 = nearby_source["THETA_J2000"]

        # define a scaling-up factor for the ellipse size
        scaling_factor = 5

        # Convert the SExtractor parameters to pixel units (assuming a pixel scale of 0.2 arcsec/pixel)
        a_pix = a_world / (0.2 / 3600) # Convert from degrees to arcseconds, then to pixels
        b_pix = b_world / (0.2 / 3600) # Convert from degrees to arcseconds, then to pixels

        # Raise a warning if any of the parameters are NaN
        if np.isnan(a_world) or np.isnan(b_world) or np.isnan(theta_j2000):
            print(f"Warning: NaN value found in SExtractor parameters for nearby source ID {nearby_source_id} "
                  f"at RA={ra}, DEC={dec}. Plotting circle using ISOAREAF_IMAGE instead")
            a_pix_hst = np.sqrt(nearby_source["ISOAREAF_IMAGE"] / np.pi) # Convert ISOAREAF_IMAGE to an equivalent radius
            b_pix_hst = a_pix_hst # Use the same value for b_pix to create a circle
            # Convert from HST to MUSE pixel scale
            a_pix = a_pix_hst * (0.03 / 0.2)
            b_pix = b_pix_hst * (0.03 / 0.2)
            theta_j2000 = 0 # Set the angle to 0 for a circle

        position = SkyCoord(ra=ra*u.degree, dec=dec*u.degree)
        position_pix = muse_img.wcs.wcs.world_to_pixel(position)

        ax.scatter(position_pix[0], position_pix[1], color="red", marker="x") # Mark the center of the nearby source with a red "x"

        # Create an ellipse patch and add it to the plot
        ellipse = Ellipse((position_pix[0], position_pix[1]), 
                          width=a_pix * scaling_factor, height=b_pix * scaling_factor, 
                          angle=theta_j2000, edgecolor="red", facecolor="none")
        ax.add_patch(ellipse)

    plt.show()

In [ ]:
muse_img.wcs?